# Loading data

Every MiTo function works on one object, the **Allele Frequency Matrix** (AFM): an
`AnnData` of cells x MT-SNVs holding molecule counts. This page shows what it contains,
and how to build it from the output of each pre-processing tool:

| Assay | Pre-processing | Call |
|---|---|---|
| MAESTER | maegatk ([nf-MiTo](https://github.com/andrecossa5/nf-MiTo)) | `make_afm(..., scLT_system='MAESTER', pp_method='maegatk')` |
| mtscATAC | mgatk (nf-MiTo support coming) | `make_afm(..., scLT_system='mtscATAC', pp_method='mgatk')` |
| ReDeeM | RedeemV | `make_afm(..., scLT_system='ReDeeM', pp_method='redeem-v')` |
| any | your own counts tables | `make_afm_from_counts(counts, coverage)` |

The inputs below are small real subsets (50 cells each) of the
[Benchmark](benchmark.ipynb) datasets, in each tool's own file layout.

In [1]:
import logging
import pandas as pd
import mito as mt

logging.getLogger().setLevel(logging.WARNING)   # MiTo logs every stage at INFO level

## MAESTER: maegatk tables

maegatk (as published by nf-MiTo) writes one table per base, `A.txt.gz` ... `T.txt.gz`
(position, cell, forward and reverse UMI counts, with their base qualities), and
`coverage.txt.gz` (position, cell, coverage). An optional metadata table restricts the
AFM to annotated cells; with `sample=`, cell names become `{barcode}_{sample}`.

In [2]:
!ls data/maester_MDA_clones

A.txt.gz  C.txt.gz  cells_meta.csv  coverage.txt.gz  G.txt.gz  T.txt.gz


In [3]:
afm = mt.io.make_afm(
    'data/maester_MDA_clones',
    path_meta='data/maester_MDA_clones/cells_meta.csv', sample='MDA_clones',
    scLT_system='MAESTER', pp_method='maegatk',
)
afm

AnnData object with n_obs × n_vars = 50 × 4228
    obs: 'GBC', 'mean_site_coverage', 'median_target_site_coverage', 'frac_target_site_covered'
    var: 'pos', 'ref', 'alt', 'quality'
    uns: 'scLT_system', 'pp_method'
    layers: 'AD', 'DP', None (.X)

## Anatomy of an AFM

| Slot | Content |
|---|---|
| `afm.X` | allele frequency, AD / DP (sparse, float32) |
| `afm.layers['AD']` | alternative-allele molecules (sparse, int) |
| `afm.layers['DP']` | coverage of the variant's **site**, for every cell (dense, int) |
| `afm.var` | `pos`, `ref`, `alt` and, where the tool reports it, base `quality` |
| `afm.obs` | the cell metadata, plus coverage metrics used by `mt.pp.filter_cells` |
| `afm.uns` | `scLT_system`, `pp_method`; later, every step's parameters under `'mito'` |

Variants are named `{pos}_{ref}>{alt}`. One alternative allele is kept per site, the
dominant one (`multi_allelic_sites=True` keeps them all).

DP is the coverage of the site, not of the alternative allele: a cell covered 100x with
no alternative molecule has AD = 0 and DP = 100, which is evidence that it does *not*
carry the variant, while DP = 0 means it was not seen at all. Here, the cells of one clone
carry `7016_C>T` at ~15%; cells of the other clones are covered and carry none:

In [4]:
v = '7016_C>T'          # a marker of one of the lentiviral clones
j = afm.var_names.get_loc(v)
df = pd.DataFrame({
    'AD': afm.layers['AD'][:, j].toarray().ravel(),
    'DP': afm.layers['DP'][:, j],
    'AF': afm.X[:, j].toarray().ravel().round(3),
    'clone (GBC)': afm.obs['GBC'].str[:8].values,
}, index=afm.obs_names.str.replace('_MDA_clones', ''))
pd.concat([df[df['AD'] > 0].head(5), df[df['AD'] == 0].head(3)])

,AD,DP,AF,clone (GBC)
GAGTGTTTCACTTCTA,8,203,0.039,CCGTCACT
GAGATGGTCCTCTCGA,57,302,0.189,CCGTCACT
TAGTGCAGTACTCGAT,33,152,0.217,CCGTCACT
CAGGTATCACTAACCA,71,1157,0.061,CCGTCACT
ATCACTTTCGTCCTCA,72,593,0.121,CCGTCACT
CACGGGTCAAAGACTA,0,776,0.000,CGTTTCGC
CTGCATCAGAATTTGG,0,186,0.000,CGTTTCGC
GAGGGATCACGACAAG,0,360,0.000,CGTTTCGC


In [5]:
afm.var.head()

,pos,ref,alt,quality
73_A>G,73,A,G,45.0
153_A>G,153,A,G,45.0
195_T>C,195,T,C,45.0
263_A>G,263,A,G,45.0
310_T>C,310,T,C,45.0


Every MiTo step records its parameters in `afm.uns['mito']`, so an AFM carries its own
history:

In [6]:
mt.pp.filter_cells(afm, cell_filter='filter2')
afm.uns['mito']

{'version': '2.0.0',
 'filter_cells': {'cell_filter': 'filter2',
  'median_cov_target': 25,
  'min_perc_covered_sites': 0.75,
  'n_cells_in': 50,
  'n_cells_out': 32}}

## mtscATAC: mgatk tables

mgatk writes the same per-base tables (`<sample>.A.txt.gz`, ..., `<sample>.coverage.txt.gz`,
usually under `final/`), but of **reads**, not molecules: mtscATAC has no UMIs. Two
consequences for genotyping: a molecule whose fragment is short is read by both mates and
counted twice, and a sequencing error is one read on one strand.

`make_afm` turns reads into molecules from the strand counts alone. It estimates the reads
per molecule *f* from how forward and reverse reads balance at each site, divides coverage
by *f*, and counts as alternative molecules those seen by both mates, min(forward,
reverse), when most molecules are (otherwise, alternative reads / *f*).

In [7]:
atac = mt.io.make_afm(
    'data/mtscATAC_pL1000', path_meta='data/mtscATAC_pL1000/cells_meta.csv',
    scLT_system='mtscATAC', pp_method='mgatk',
)
atac.uns['molecules']

{'reads_per_molecule': 1.6766784857939832,
 'frac_both_mates': 0.6766784857939832,
 'alt_rule': 'both_strands'}

`molecules=False` keeps raw read counts. Only use them with a genotyping model of your
own: MiTo's assumes molecules.

## ReDeeM: RedeemV output

RedeemV writes consensus molecules: `RawGenotypes.<level>.StrandBalance` (one row per
alternative molecule) and `QualifiedTotalCts` (coverage per cell and site, at four
consensus stringencies). `treshold` picks the level (`'Sensitive'` by default, as
MitoDrift). Basecalls within `edge_trim` = 9 bp of a fragment end are dropped, as redeemR
does: fragment ends concentrate damage. Both files may be gzipped.

In [8]:
redeem = mt.io.make_afm(
    'data/redeem_pL1000', path_meta='data/redeem_pL1000/cells_meta.csv',
    scLT_system='ReDeeM', pp_method='redeem-v',
)
redeem

AnnData object with n_obs × n_vars = 50 × 133
    obs: 'LARRY', 'mean_site_coverage'
    var: 'pos', 'ref', 'alt'
    uns: 'scLT_system', 'pp_method'
    layers: 'AD', 'DP', None (.X)

## Any pipeline: counts and coverage tables

If your counts come from elsewhere, two long tables are enough:

* `counts`: one row per cell and variant with at least one alternative molecule -
  `cell`, `pos` (1-based), `alt`, `AD`, optionally `ref` (checked against the reference)
  and `quality`;
* `coverage`: one row per cell and **covered site** - `cell`, `pos`, `DP`. Include the
  covered sites without alternative molecules: that is the evidence of absence.

In [9]:
counts = pd.DataFrame({
    'cell': ['c1', 'c2', 'c2', 'c3'],
    'pos':  [3243, 3243, 9537, 9537],
    'alt':  ['G',  'G',  'T',  'T'],
    'AD':   [12,   7,    3,    15],
})
coverage = pd.DataFrame({
    'cell': ['c1', 'c2', 'c3'] * 2,
    'pos':  [3243] * 3 + [9537] * 3,
    'DP':   [80, 60, 75, 40, 52, 66],
})
toy = mt.io.make_afm_from_counts(counts, coverage)
toy.to_df(layer='AD').join(toy.to_df(layer='DP'), rsuffix=' (DP)')

,3243_A>G,9537_C>T,3243_A>G (DP),9537_C>T (DP)
c1,12,0,80,40
c2,7,3,60,52
c3,0,15,75,66


The reference base comes from rCRS (`ref=` takes a FASTA for other genomes), and
`cell_meta=` restricts and annotates the cells, as `path_meta` does for `make_afm`.

## AFMs from earlier MiTo versions

AFMs written by MiTo < 0.3 kept a masked DP and several alleles per site:
`mt.io.migrate_afm(afm)` converts them to the current object.

Next: [Getting started](getting_started.ipynb) runs the pipeline on an AFM.